# Question 23 - 28. Find the Index of the First Occurrence in a String

Given two strings `needle` and `haystack`, return the index of the first occurrence of `needle` in `haystack`, or `-1` if `needle` is not part of `haystack`.

**Example 1:**

    Input: haystack = "sadbutsad", needle = "sad"
    Output: 0
    Explanation: "sad" occurs at index 0 and 6.
    The first occurrence is at index 0, so we return 0.

**Example 2:**

    Input: haystack = "leetcode", needle = "leeto"
    Output: -1
    Explanation: "leeto" did not occur in "leetcode", so we return -1.

**Constraints:**

- `1 <= haystack.length, needle.length <= 10^4`
- `haystack` and `needle` consist of only lowercase English characters.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** String search: sliding brute force → KMP (never re-read the haystack)

📘 **Revise first:** [Part 5.3 · searching in strings](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every start) | O(n · m) | O(1) |
| 2️⃣ Optimized (KMP) | O(n + m) | O(m) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Find where a short word (the **needle**) first appears inside a longer text (the **haystack**). Return its starting position, or `-1` if it never appears. This is what "Find" (Ctrl+F) does in a text editor.

```
haystack = "sadbutsad", needle = "sad"  → 0
haystack = "leetcode",  needle = "leeto" → -1
```

Let `n` = length of the haystack and `m` = length of the needle.

**Everyday picture: sliding a stencil along a line of text.** Put the stencil (needle) at the start of the text and check letter by letter. If it doesn't match, slide the stencil one step right and check again.

That's the brute force, and it's often fine. But on nasty inputs it **forgets** what it just read: after matching 4 letters and failing on the 5th, it slides one step and **re-reads** 3 of those same letters. KMP is the trick that **remembers**, so it never re-reads a single letter of the text.

### Step 0 · Clarify before coding

🗣️ *"This is substring search. I'll mention that Python's haystack.find(needle) does it in one call, but I assume you'd like me to implement the search myself."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Can I use `str.find`? | Usually **no**: they want the algorithm. | Mention it anyway; it shows you know the library. |
| Empty needle? | Not possible here (length ≥ 1). | Conventionally the answer would be 0. |
| Needle longer than haystack? | Possible → `-1`. | |
| Size? | Both up to 10⁴. | Brute force worst case: 10⁸ comparisons. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** for every starting position `i` in the haystack, compare the needle letter by letter. Stop at the first mismatch, and return `i` on a full match.

🗣️ *"The direct approach tries every start position and compares the needle character by character, stopping at the first mismatch. On normal text, mismatches happen fast, so it's quick. But the worst case is O(n·m)."*

**When is it slow?** Text `"aaaaaaaaab"` with needle `"aaab"`. At every start we match `aaa`, then fail on `b`, slide one step, and match the **same** `aa` again. Each start does almost m comparisons → **n × m**.

In [1]:
class SolutionBrute:
    def strStr(self, haystack: str, needle: str) -> int:
        n, m = len(haystack), len(needle)
        for i in range(n - m + 1):                  # every possible start
            j = 0
            while j < m and haystack[i + j] == needle[j]:
                j += 1
            if j == m:                              # matched the whole needle
                return i
        return -1

### Step 2 · Optimize: the "aha" moment (KMP)

KMP is named after Knuth, Morris and Pratt. Here's the idea in plain words.

Say I've matched the first `j` letters of the needle, and then the next letter doesn't match. **I already know exactly what those last `j` letters of the text were: they're `needle[:j]`.** So instead of sliding back and re-reading them, I can ask:

> *"Is there a **shorter** start of the needle that is also an **ending** of what I just matched? If so, I can keep that much matched and continue."*

**Example:** needle `"ababc"`. I matched `abab` and failed on the next letter. `abab` **ends** with `ab`, which is also how the needle **starts**. So I don't restart from zero. I pretend I've already matched `ab` and continue from there, **without moving backwards in the text**.

**Precompute this once for the needle.** It's called the **LPS array** ("longest prefix that is also a suffix"):

> `lps[j]` = for the first `j + 1` letters of the needle, the length of the longest piece that is **both** a proper start and an ending.

Then the search is:
- Letters match → advance both.
- Mismatch after `j` matched letters → set `j = lps[j − 1]` (keep the overlap) and try again **with the same text letter**.
- The text pointer **never moves backward**.

🗣️ *"Brute force throws away what it just matched. KMP precomputes, for each prefix of the needle, the longest proper prefix that's also a suffix. On a mismatch, instead of restarting, I drop back to that length, because I know those characters already match, and I never move backwards in the haystack. Building the table is O(m), the scan is O(n), so O(n + m) total with O(m) extra space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| `haystack.find(needle)` | ✅ in real code | Fast C code. But the interview wants the algorithm. |
| Brute force sliding | ⚠️ good opener | O(n·m) worst case. Simple, and fine for typical text. |
| Rabin–Karp (rolling hash) | ✅ alternative | Turns each window into a number and compares numbers. Expected O(n+m), but hash collisions mean you must double-check matches. |
| **KMP (LPS table)** | ✅ **guaranteed** | Worst case O(n+m), no luck involved. Costs an O(m) table. |
| Z-algorithm | ✅ alternative | Also O(n+m). A different way to build similar information. |
| Boyer–Moore | ⚠️ | Very fast in practice, but complicated to write correctly in an interview. |

In [2]:
class Solution:
    def strStr(self, haystack: str, needle: str) -> int:
        n, m = len(haystack), len(needle)
        if m == 0:
            return 0

        # Build the LPS table: lps[i] = length of the longest proper prefix of
        # needle[:i+1] that is also a suffix of it.
        lps = [0] * m
        length = 0                              # length of the current matching prefix
        for i in range(1, m):
            while length > 0 and needle[i] != needle[length]:
                length = lps[length - 1]        # fall back to a shorter overlap
            if needle[i] == needle[length]:
                length += 1
            lps[i] = length

        # Scan the haystack; i only ever moves forward.
        j = 0                                   # how many needle letters are matched
        for i in range(n):
            while j > 0 and haystack[i] != needle[j]:
                j = lps[j - 1]                  # keep the overlap instead of restarting
            if haystack[i] == needle[j]:
                j += 1
            if j == m:                          # full match ends at i
                return i - m + 1
        return -1

### Step 3 · Toy example walkthrough (tell it like a story)

**Part 1: build the LPS table for `needle = "ababc"`**

| i | first i+1 letters | longest piece that is both a start and an end | lps[i] |
|---|---|---|---|
| 0 | `a` | none ("proper" means not the whole thing) | 0 |
| 1 | `ab` | none | 0 |
| 2 | `aba` | `a` | 1 |
| 3 | `abab` | `ab` | 2 |
| 4 | `ababc` | none | 0 |

`lps = [0, 0, 1, 2, 0]`

**Part 2: search in `haystack = "abababc"`**

| i | text letter | j before | compare | what happens | j after |
|---|---|---|---|---|---|
| 0 | a | 0 | a = a | match | 1 |
| 1 | b | 1 | b = b | match | 2 |
| 2 | a | 2 | a = a | match | 3 |
| 3 | b | 3 | b = b | match | 4 |
| 4 | a | 4 | a ≠ c | **mismatch** → j = lps[3] = 2; now compare a with needle[2] = a ✅ | 3 |
| 5 | b | 3 | b = b | match | 4 |
| 6 | c | 4 | c = c | match | **5 = m** → found at 6 − 5 + 1 = **2** ✅ |

**Narrated:** "I match a-b-a-b, then expect a 'c' but see an 'a'. Brute force would go back to text position 1 and start again. KMP says: the 'abab' I just read ends in 'ab', which is how the needle begins, so I keep 2 letters matched and continue right here. The 'a' matches the needle's third letter, then b and c follow. Found at index 2, and I never moved backwards in the text."

In [3]:
cases = [
    ("sadbutsad", "sad", 0),
    ("leetcode", "leeto", -1),
    ("abababc", "ababc", 2),
    ("a", "a", 0),
    ("abc", "c", 2),
    ("aaa", "aaaa", -1),
    ("aaaaab", "aab", 3),
    ("mississippi", "issip", 4),
]
for h, nd, expected in cases:
    assert SolutionBrute().strStr(h, nd) == expected
    assert Solution().strStr(h, nd) == expected
    assert h.find(nd) == expected
    print(h, nd, "->", expected)

import random
for _ in range(2000):
    h = "".join(random.choices("ab", k=random.randint(1, 15)))
    nd = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert Solution().strStr(h, nd) == h.find(nd), (h, nd)
print("All tests passed ✅")

sadbutsad sad -> 0
leetcode leeto -> -1
abababc ababc -> 2
a a -> 0
abc c -> 2
aaa aaaa -> -1
aaaaab aab -> 3
mississippi issip -> 4
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Brute force | O(n · m) worst | O(1) | May re-read the same text letters many times. |
| Rabin–Karp | O(n + m) expected | O(1) | Compares numbers instead of strings; rare collisions. |
| **KMP** | **O(n + m)** worst | **O(m)** | Never re-reads the text; a table the size of the needle. |

**Why is KMP O(n + m), even with a `while` inside a `for`?** Think of `j` as a counter. It goes **up by at most 1** for each text letter, so at most n times in total. Every fallback makes it **go down**, and it can never go below 0. You can't go down more times than you went up, so the total number of fallbacks is at most n. The search is O(n). The same argument makes building the table O(m).

**Why O(m) memory?** The LPS table has one number per needle letter.

**Edge cases to mention:** needle longer than haystack, match at the very end, overlapping partial matches (`"aaaaab"` / `"aab"`), single letters.

**Likely follow-ups:**
- *Find all occurrences?* → On a full match, record it, set `j = lps[j − 1]`, and keep going.
- *Repeated Substring Pattern (LC 459)?* → The last LPS value reveals the smallest repeating unit.

---

#### 🗣️ Full interview script (about 60 seconds)

*"I'm looking for the first occurrence of the needle in the haystack. In production I'd use haystack.find, but I'll implement it.*

*Brute force tries every start position and compares character by character; that's O(n·m) in the worst case, for inputs like many a's followed by a b.*

*KMP avoids re-reading the haystack. I precompute, for every prefix of the needle, the length of the longest proper prefix that is also a suffix. During the scan, when I hit a mismatch after matching j characters, I already know those j characters, so I fall back to lps[j−1] matched characters instead of zero, and retry the same haystack character. The haystack index only moves forward.*

*Building the table is O(m) and scanning is O(n), using an amortised argument: j rises at most once per character, so it can't fall more than that. Total O(n + m) time and O(m) space."*